In [8]:
import pandas as pd
import kagglehub


path = kagglehub.dataset_download("fazilbtopal/misspelled-words")

df=pd.read_csv(f"{path}/misspelled.csv")
df.drop("Unnamed: 0", axis=1, inplace=True)
df.rename(columns={"label": "word","input": "misspelled"}, inplace=True)
df.to_csv("../old_data/misspelled/common_misspelled.csv")

In [9]:
import random
from multiprocessing import Pool, cpu_count
from tqdm import tqdm

# Set random seed for reproducibility
random.seed(999)

# QWERTY layout and their neighbors 
NEIGHBORS = {
    'q': 'was', 'w': 'qeasd', 'e': 'wrsdf', 'r': 'etdfg', 't': 'ryfgh',
    'y': 'tughj', 'u': 'yihjk', 'i': 'uojkl', 'o': 'ipkl', 'p': 'ol',
    'a': 'qwszx', 's': 'qwedazx', 'd': 'werfscx', 'f': 'ertgvds', 'g': 'rtyhbdf',
    'h': 'tyujngb', 'j': 'yuhmkn', 'k': 'uijlm', 'l': 'opk',
    'z': 'asx', 'x': 'zsdc', 'c': 'xdfv', 'v': 'cfgb', 'b': 'vghn',
    'n': 'bhjm', 'm': 'njk'
}

_misspellings_df = pd.read_csv("../old_data/misspelled/common_misspelled.csv")
MISSPELLINGS = dict(zip(_misspellings_df['word'].str.lower(), _misspellings_df['misspelled']))

def key_swap(word):
    """Swap a character with a neighboring key on QWERTY keyboard."""
    if not word:
        return word
    idx = random.randrange(len(word))
    char = word[idx].lower()
    neighbors = NEIGHBORS.get(char)
    if neighbors:
        return word[:idx] + random.choice(neighbors) + word[idx + 1:]
    return word


def char_swap(word):
    """Swap two adjacent characters."""
    if len(word) < 2:
        return word
    idx = random.randrange(len(word) - 1)
    return word[:idx] + word[idx + 1] + word[idx] + word[idx + 2:]


def char_remove(word):
    """Remove a random character from the word."""
    if len(word) < 3:
        return word
    idx = random.randrange(len(word))
    return word[:idx] + word[idx + 1:]


def punc_misplace(word):
    """Add or remove punctuation at the end of a word."""
    if word[-1].isalnum():
        return word + random.choice(',!?.')
    return word[:-1]


def common_misspell(word):
    """Replace word with a common misspelling if available."""
    return MISSPELLINGS.get(word.lower(), word)


# Pre-define attack functions tuple for faster access
ATTACK_TYPES = (key_swap, char_swap, char_remove, punc_misplace, common_misspell)


def _init_worker(seed):
    """Initialize worker process with seed."""
    random.seed(seed)


def _spelling_attack(text, target_rate=0.20):
    """Apply attacks on target_rate% of words in the text."""
    if not isinstance(text, str) or not text.strip():
        return text
    
    words = text.split()
    num_words = len(words)
    if num_words == 0:
        return text

    num_to_attack = max(1, int(num_words * target_rate))
    
    target_idx = random.sample(range(num_words), num_to_attack)
    attack_idx = random.choices(range(5), k=num_to_attack)
    
    # Apply attacks
    for i, idx in enumerate(target_idx):
        words[idx] = ATTACK_TYPES[attack_idx[i]](words[idx])

    return " ".join(words)


def _attack_wrapper(args):
    """Wrapper for multiprocessing."""
    text, target_rate = args
    return _spelling_attack(text, target_rate)


def apply_spelling_attack(df, text_column, target_rate=0.20, n_workers=None, seed=999):
    if n_workers is None:
        n_workers = max(1, cpu_count() - 1)
    
    texts = df[text_column].tolist()
    args = [(text, target_rate) for text in texts]
    
    with Pool(n_workers, initializer=_init_worker, initargs=(seed,)) as pool:
        results = list(tqdm(
            pool.imap(_attack_wrapper, args, chunksize=100),
            total=len(texts),
            desc=f"Attacking {text_column}"
        ))
    
    return results

In [11]:
from sklearn.model_selection import train_test_split
import os

df=pd.read_csv("../old_data/original_DB.csv")

_, x_test= train_test_split(
    df, 
    test_size=0.2,
    random_state=999, 
)
TARGET_RATES = [0.05, 0.10, 0.15, 0.20]

x_test.drop("prompt", axis=1, inplace=True)

for rate in TARGET_RATES:
    print(f"--- Running Attack at {rate*100}% ---")
    
    # 1. Apply Attack 
    x_test_attacked = x_test.copy()
    for col in x_test_attacked.columns:
        x_test_attacked[col] = apply_spelling_attack(x_test_attacked, col, target_rate=rate, seed=999)
    
    # save attacked dataset for reference
    attacked_path = f"../old_data/misspelled/{int(rate*100)}/attacked_test.csv"
    # make the directory if it doesn't exist
    
    os.makedirs(os.path.dirname(attacked_path), exist_ok=True)
    x_test_attacked.to_csv(attacked_path, index=False)
    print(f"Attacked dataset saved to {attacked_path}")


--- Running Attack at 5.0% ---


Attacking GPT_4-o: 100%|██████████| 1465/1465 [00:00<00:00, 29852.19it/s]


Attacked dataset saved to ../old_data/misspelled/5/attacked_test.csv
--- Running Attack at 10.0% ---


Attacking GPT_4-o: 100%|██████████| 1465/1465 [00:00<00:00, 29223.80it/s]


Attacked dataset saved to ../old_data/misspelled/10/attacked_test.csv
--- Running Attack at 15.0% ---


Attacking GPT_4-o: 100%|██████████| 1465/1465 [00:00<00:00, 18170.64it/s]


Attacked dataset saved to ../old_data/misspelled/15/attacked_test.csv
--- Running Attack at 20.0% ---


Attacking GPT_4-o: 100%|██████████| 1465/1465 [00:00<00:00, 11926.04it/s]


Attacked dataset saved to ../old_data/misspelled/20/attacked_test.csv


In [ ]:
x_test.columns

Index(['Human_story', 'gemma-2-9b', 'mistral-7B', 'qwen-2-72B', 'llama-8B',
       'accounts/yi-01-ai/models/yi-large', 'GPT_4-o'],
      dtype='object')